In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/hibaadli30018/bert-base-uncased/__results__.html
/kaggle/input/notebooks/hibaadli30018/bert-base-uncased/__notebook__.ipynb
/kaggle/input/notebooks/hibaadli30018/bert-base-uncased/__output__.json
/kaggle/input/notebooks/hibaadli30018/bert-base-uncased/custom.css
/kaggle/input/datasets/abhishek/bert-base-uncased/config.json
/kaggle/input/datasets/abhishek/bert-base-uncased/pytorch_model.bin
/kaggle/input/datasets/abhishek/bert-base-uncased/vocab.txt
/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Imports

In [45]:
from sklearn.preprocessing import LabelEncoder

from transformers import pipeline,AutoTokenizer,AutoModel,AutoModelForCausalLM,AutoModelForMultipleChoice,TrainingArguments,Trainer
import transformers
import torch

from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

In [3]:
train=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

# Question 1

In [4]:
le=LabelEncoder()
train['label']=le.fit_transform(train['answer'])
train.head()

,id,prompt,A,B,C,D,E,answer,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,0
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,2
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,0


In [5]:
print("Encoded numeric label for the row at index 150:",train.iloc[150]['label'])

Encoded numeric label for the row at index 150: 2


# Question 2

In [6]:
prompt=train['prompt'][0]
option_B=train['B'][0]
aug_str=str(prompt) + " [SEP] " + str(option_B)
print("Character length of formatted input string:",len(aug_str))

Character length of formatted input string: 407


# Question 3

In [7]:
tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased",)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
formatted_inputs=[(str(prompt) + " [SEP] " + str(train[opt][0])) for opt in "ABCDE"]
tokens=tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [9]:
mcq_tokens={k:v.unsqueeze(0) for k,v in tokens.items()}
print("Value of Second Dimenstion:",mcq_tokens['input_ids'].shape[1])

Value of Second Dimenstion: 5


# Question 4

In [10]:
rows_16_inputs=[]
for _,row in train.iloc[:16].iterrows():
    formatted_inputs=[(str(row['prompt']) + " [SEP] " + str(row[opt])) for opt in "ABCDE"]
    rows_16_inputs.extend(formatted_inputs)
tokens_16=tokenizer(
    rows_16_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
tokens_16["input_ids"].shape

torch.Size([80, 128])

In [11]:
input_ids=tokens_16["input_ids"].reshape(16, 5, 128)

print("Total token positions",input_ids.numel())

Total token positions 10240


# Question 5

In [14]:
mcq_model=AutoModelForMultipleChoice.from_pretrained("/kaggle/input/datasets/abhishek/bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: /kaggle/input/datasets/abhishek/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoin

In [26]:
output=mcq_model(**mcq_tokens,labels=torch.tensor([train['label'][0]]))
print("Number of Logits:",output.logits.shape)

Number of Logits: torch.Size([1, 5])


# Question 6

In [30]:
print("Number of Dimenstions:",output.loss.dim())

Number of Dimenstions: 0


# Question 7

In [34]:
lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
mcq_lora_model=get_peft_model(mcq_model, lora_config)

In [35]:
trainable_params=sum(p.numel() for p in mcq_lora_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

Trainable parameters: 295,681


# Question 8

In [47]:
hf_dataset=Dataset.from_pandas(train.iloc[:100])
hf_dataset

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'label'],
    num_rows: 100
})

In [48]:
def preprocess(example):
    choices=["A", "B", "C", "D", "E"]

    inputs=[str(example["prompt"]) + " [SEP] " + str(example[c]) for c in choices]

    tokens=tokenizer(
        inputs,
        padding="max_length",
        truncation=True,
        max_length=64
    )

    return {
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"],
        "labels": example["label"]
    }

In [49]:
hf_dataset=hf_dataset.map(preprocess)
hf_dataset=hf_dataset.remove_columns([c for c in hf_dataset.column_names if c not in ["input_ids", "attention_mask", "labels"]])
hf_dataset

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})

In [50]:
print("Tokenized IDs Present in Input IDs:",len(hf_dataset[0]["input_ids"]))

Tokenized IDs Present in Input IDs: 5


# Question 9

In [51]:
train_args=TrainingArguments(
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1,
    max_steps = 4
)

In [66]:
trainer=Trainer(
    model=mcq_lora_model,
    args=train_args,
    train_dataset=hf_dataset.select(range(32))
)

In [67]:
trainer.train()
print("Global Step:", trainer.state.global_step)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Global Step: 4


# Question 10

In [68]:
mcq_lora_model.eval()

with torch.no_grad():
    outputs=mcq_lora_model(**mcq_tokens)

logits=outputs.logits
probs=torch.softmax(logits, dim=1)
print(f"Probability of Option E: {probs[0, 4].item():.4f}")

Probability of Option E: 0.2033
